# Node2Vec Features + Graph Transformer (Graphormer) + MFC + TopoReg

This notebook extends the Node2Vec + MFC + TopoReg baseline by replacing the
GCN backbone encoder with a **Graph Transformer** encoder combining full
multi-head self-attention with a Graphormer-style structural bias.

## What changes vs. the GCN baseline

The only change to the pipeline is the encoder inside `GAEMF`. Instead of
`GraphConvSparse` (symmetric-normalised `adj @ x @ W`), the encoder is now
`GraphTransformerConv`. It computes full N×N multi-head self-attention and
injects the normalised adjacency as a per-head learnable edge bias:
`attn[i,j,h] += adj[i,j] * edge_bias[h]`. This unifies Graphormer (structural
encoding) and SAN (global attention + local structure) in a single layer.
`num_heads` is auto-selected as the largest divisor of `encoded_space_dim` that
is ≤ 8. All other components — node2vec feature extraction, MFC clustering,
community topology, WRCF filtration, Wasserstein TopoReg loss, and evaluation —
are identical to the baseline.

## Why Graph Transformer on top of node2vec features

Node2vec captures local random-walk structure. The Graph Transformer adds
global pairwise attention, letting every node attend to every other regardless
of hop distance. The structural bias from the adjacency anchors that global
attention back to the observed edges, preventing it from ignoring graph topology.

**Dense adjacency is kept intentionally.** The normalised adjacency matrix
(`adj_norm_dense`) must remain a dense `torch.Tensor` for the structural bias
computation `adj.unsqueeze(-1) * edge_bias` to broadcast correctly over heads.
The attention itself is O(N²·H) by design and also requires the full matrix.

Saves to `Output/node2vec_transformer_mfc_toporeg`.

## References

1. Dexu Kong, Anping Zhang, Yang Li. *Learning Persistent Community Structures in Dynamic Networks via Topological Data Analysis*. AAAI 2024. Code/paper basis for `MFC + TopoReg`.
   Link: `Original Paper.pdf`

2. Aditya Grover, Jure Leskovec. *node2vec: Scalable Feature Learning for Networks*. KDD 2016.
   DOI: https://doi.org/10.1145/2939672.2939754
   KDD page: https://www.kdd.org/kdd2016/subtopic/view/node2vec-scalable-feature-learning-for-networks

3. Sadamori Kojaku, Filippo Radicchi, Yong-Yeol Ahn, Santo Fortunato. *Network community detection via neural embeddings*. Nature Communications, 2024.
   Link: https://www.nature.com/articles/s41467-024-52355-w

4. Petar Velickovic et al. *Deep Graph Infomax*. ICLR 2019.
   Link: https://arxiv.org/abs/1809.10341

5. Zhenyu Hou et al. *GraphMAE: Self-Supervised Masked Graph Autoencoders*. KDD 2022.
   Link: https://arxiv.org/abs/2205.10803

6. Shantanu Thakoor et al. *Large-Scale Representation Learning on Graphs via Bootstrapping (BGRL)*. ICLR 2022.
   Link: https://arxiv.org/abs/2102.06514

The code below implements item 2 as the first concrete improvement because it is simpler than graph self-supervised pretraining, keeps the setting attribute-free, and directly targets the main bottleneck in the current reproduction: weak node inputs.


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import json
import pickle
import random

try:
    import gudhi as gd
    from gudhi.wasserstein import wasserstein_distance
except ModuleNotFoundError:
    gd = None
    wasserstein_distance = None

import networkx as nx
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    from gensim.models import Word2Vec
except ModuleNotFoundError:
    Word2Vec = None
from sklearn import metrics
from sklearn.cluster import KMeans


graph_pkl = ["enron", "highschool", "DBLP", "Cora", "DBLPdyn"]
label_num_dic = {"Cora": 10, "enron": 7, "highschool": 9, "DBLP": 15, "DBLPdyn": 14}


def require_gudhi() -> None:
    if gd is None or wasserstein_distance is None:
        raise ModuleNotFoundError("gudhi is required. Install it before running this notebook.")


def require_gensim() -> None:
    if Word2Vec is None:
        raise ModuleNotFoundError("gensim is required. Install it before running this notebook.")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def resolve_device(device_name: Optional[str]) -> torch.device:
    if device_name:
        return torch.device(device_name)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def resolve_n_cluster(config: "ExperimentConfig", label_count: int) -> int:
    if config.n_cluster is not None:
        return config.n_cluster
    if config.dataset_name == "DBLPdyn":
        return max(label_count, label_count * config.unknown_k_multiplier)
    return label_count


def should_use_argmax_q(config: "ExperimentConfig") -> bool:
    if config.eval_use_argmax_q is not None:
        return config.eval_use_argmax_q
    return config.dataset_name == "DBLPdyn"


def glorot_init(input_dim: int, output_dim: int) -> nn.Parameter:
    init_range = np.sqrt(6.0 / (input_dim + output_dim))
    initial = torch.rand(input_dim, output_dim) * 2 * init_range - init_range
    return nn.Parameter(initial, requires_grad=True)


class GraphConvSparse(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, adj: torch.Tensor, activation=F.leaky_relu) -> None:
        super().__init__()
        self.weight = glorot_init(input_dim, output_dim)
        self.adj = adj
        self.activation = activation

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.mm(x, self.weight)
        x = torch.mm(self.adj, x)
        return self.activation(x)


class GraphTransformerConv(nn.Module):
    """Graph Transformer encoder with Graphormer-style structural bias.
    Implements full multi-head self-attention where the normalised adjacency
    injects a per-head learnable edge bias into every attention score:
        attn[i,j,h] += adj[i,j] * edge_bias[h]
    This covers both Graphormer (structural/spatial encoding) and SAN
    (global attention + local structure) design principles in a single layer.
    num_heads is auto-selected as the largest divisor of output_dim that is <= 8.
    Input features are node2vec structural embeddings (structural_feature_dim).
    NOTE: attention is O(N^2 * H) — dense adjacency is kept as required.
    """
    def __init__(self, input_dim: int, output_dim: int, adj: torch.Tensor,
                 num_heads: int = 0, activation=F.leaky_relu) -> None:
        super().__init__()
        if num_heads <= 0:
            num_heads = max(h for h in range(1, min(output_dim, 8) + 1) if output_dim % h == 0)
        assert output_dim % num_heads == 0, "output_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim  = output_dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.Q         = nn.Linear(input_dim, output_dim, bias=False)
        self.K         = nn.Linear(input_dim, output_dim, bias=False)
        self.V         = nn.Linear(input_dim, output_dim, bias=False)
        self.out_proj  = nn.Linear(output_dim, output_dim)
        self.edge_bias = nn.Parameter(torch.zeros(num_heads))
        self.adj = adj
        self.activation = activation

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        N = x.size(0)
        Q = self.Q(x).view(N, self.num_heads, self.head_dim)
        K = self.K(x).view(N, self.num_heads, self.head_dim)
        V = self.V(x).view(N, self.num_heads, self.head_dim)
        attn = torch.einsum('ihd,jhd->ijh', Q, K) * self.scale
        struct_bias = self.adj.unsqueeze(-1) * self.edge_bias.view(1, 1, self.num_heads)
        attn = F.softmax(attn + struct_bias, dim=1)
        out = torch.einsum('ijh,jhd->ihd', attn, V).reshape(N, -1)
        return self.activation(self.out_proj(out))



class GAEMF(nn.Module):
    def __init__(self, adj: torch.Tensor, feature_dim: int, args: "ExperimentConfig") -> None:
        super().__init__()
        self.encoder = GraphTransformerConv(feature_dim, args.encoded_space_dim, adj)
        self.cluster_centroid = glorot_init(args.n_cluster, args.encoded_space_dim)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        z = self.encoder(x)
        return F.normalize(z, p=2, dim=1)

    @staticmethod
    def normalize_rows(x: torch.Tensor) -> torch.Tensor:
        row_min = x.min(dim=1).values[:, None]
        row_max = x.max(dim=1).values[:, None]
        denom = (row_max - row_min).clamp_min(1e-12)
        x_std = (x - row_min) / denom
        return x_std / x_std.sum(dim=1, keepdim=True).clamp_min(1e-12)

    def forward(self, x: torch.Tensor, use_mf: bool):
        z = self.encode(x)
        adj_pred = torch.sigmoid(torch.matmul(z, z.t()))
        if use_mf:
            pinv_weight = torch.linalg.pinv(self.cluster_centroid)
            q = self.normalize_rows(torch.mm(z, pinv_weight))
            return adj_pred, z, q
        return adj_pred, z, None


@dataclass
class ExperimentConfig:
    dataset_name: str = "highschool"
    max_snapshots: int = 10
    encoded_space_dim: int = 30
    n_cluster: Optional[int] = None
    num_pretrain_epoch: int = 500
    num_topo_epoch: int = 500
    start_mf: int = 300
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    alpha_pretrain: float = 10.0
    alpha_topo: float = 1.0
    beta_topo: float = 10.0
    topo_cardinality: int = 32
    eval_use_argmax_q: Optional[bool] = None
    unknown_k_multiplier: int = 2
    seed: int = 7
    save_dir: str = "Output/node2vec_transformer_mfc_toporeg"
    device: Optional[str] = None

    structural_feature_dim: int = 128
    node2vec_walk_length: int = 40
    node2vec_num_walks: int = 10
    node2vec_window_size: int = 10
    node2vec_epochs: int = 20
    node2vec_p: float = 1.0
    node2vec_q: float = 2.0
    node2vec_workers: int = 1


@dataclass
class SnapshotData:
    adj_sp: sp.coo_matrix
    adj_dense: torch.Tensor
    adj_norm_dense: torch.Tensor
    features: torch.Tensor
    labels: torch.Tensor


def graph_normalization(adj: sp.coo_matrix) -> sp.coo_matrix:
    adj_ = adj + sp.eye(adj.shape[0])
    rowsum = np.array(adj_.sum(1))
    degree_mat_inv_sqrt = sp.diags(np.power(rowsum, -0.5).flatten())
    return adj_.dot(degree_mat_inv_sqrt).transpose().dot(degree_mat_inv_sqrt).tocoo()


def node2vec_cache_path(config: ExperimentConfig, snapshot_index: int) -> Path:
    feature_dir = Path(config.save_dir) / "features" / config.dataset_name
    feature_dir.mkdir(parents=True, exist_ok=True)
    signature = (
        f"d{config.structural_feature_dim}"
        f"_wl{config.node2vec_walk_length}"
        f"_nw{config.node2vec_num_walks}"
        f"_ws{config.node2vec_window_size}"
        f"_ep{config.node2vec_epochs}"
        f"_p{config.node2vec_p}"
        f"_q{config.node2vec_q}"
    )
    return feature_dir / f"snapshot_{snapshot_index}_{signature}.npy"


def node2vec_transition_probs(
    graph: nx.Graph,
    prev_node,
    current_node,
    neighbors: List,
    p: float,
    q: float,
) -> np.ndarray:
    probs = []
    for next_node in neighbors:
        weight = graph[current_node][next_node].get("weight", 1.0)
        if next_node == prev_node:
            alpha = 1.0 / max(p, 1e-12)
        elif graph.has_edge(next_node, prev_node):
            alpha = 1.0
        else:
            alpha = 1.0 / max(q, 1e-12)
        probs.append(weight * alpha)
    probs = np.asarray(probs, dtype=np.float64)
    total = probs.sum()
    if total <= 0:
        return np.full(len(neighbors), 1.0 / len(neighbors), dtype=np.float64)
    return probs / total


def generate_node2vec_walks(
    graph: nx.Graph,
    walk_length: int,
    num_walks: int,
    p: float,
    q: float,
    seed: int,
) -> List[List[str]]:
    rng = random.Random(seed)
    np_rng = np.random.default_rng(seed)
    nodes = list(graph.nodes())
    walks: List[List[str]] = []

    for walk_round in range(num_walks):
        shuffled_nodes = nodes[:]
        rng.shuffle(shuffled_nodes)
        for start in shuffled_nodes:
            walk = [start]
            while len(walk) < walk_length:
                current = walk[-1]
                neighbors = list(graph.neighbors(current))
                if not neighbors:
                    break
                if len(walk) == 1:
                    next_node = rng.choice(neighbors)
                else:
                    prev = walk[-2]
                    probs = node2vec_transition_probs(graph, prev, current, neighbors, p=p, q=q)
                    next_node = neighbors[int(np_rng.choice(len(neighbors), p=probs))]
                walk.append(next_node)
            walks.append([str(node) for node in walk])
    return walks


def compute_node2vec_features(
    graph: nx.Graph,
    snapshot_index: int,
    config: ExperimentConfig,
) -> np.ndarray:
    require_gensim()
    cache_path = node2vec_cache_path(config, snapshot_index)
    if cache_path.exists():
        return np.load(cache_path)

    walks = generate_node2vec_walks(
        graph=graph,
        walk_length=config.node2vec_walk_length,
        num_walks=config.node2vec_num_walks,
        p=config.node2vec_p,
        q=config.node2vec_q,
        seed=config.seed + snapshot_index,
    )
    model = Word2Vec(
        sentences=walks,
        vector_size=config.structural_feature_dim,
        window=config.node2vec_window_size,
        min_count=0,
        sg=1,
        hs=0,
        negative=5,
        workers=config.node2vec_workers,
        epochs=config.node2vec_epochs,
        seed=config.seed + snapshot_index,
    )

    features = np.zeros((graph.number_of_nodes(), config.structural_feature_dim), dtype=np.float32)
    for idx, node in enumerate(graph.nodes()):
        features[idx] = model.wv[str(node)]

    norms = np.linalg.norm(features, axis=1, keepdims=True)
    features = features / np.clip(norms, a_min=1e-12, a_max=None)
    np.save(cache_path, features)
    return features


def load_dynamic_dataset(config: ExperimentConfig, device: torch.device) -> Tuple[List[SnapshotData], int]:
    if config.dataset_name not in graph_pkl:
        raise ValueError(f"Unknown dataset: {config.dataset_name}")

    graph_path = Path("Data") / f"{config.dataset_name}.pkl"
    label_path = Path("Data") / f"{config.dataset_name}_label.pkl"
    if not graph_path.exists() or not label_path.exists():
        raise FileNotFoundError(
            f"Missing dataset files. Expected {graph_path} and {label_path}. "
            "Place the original dataset pickle files under Data/ before running."
        )

    with graph_path.open("rb") as handle:
        graphs = pickle.load(handle, encoding="bytes", fix_imports=True)
    with label_path.open("rb") as handle:
        labels = pickle.load(handle, encoding="bytes", fix_imports=True)

    if config.max_snapshots:
        graphs = graphs[: config.max_snapshots]
    label_count = label_num_dic[config.dataset_name]

    if config.dataset_name != "DBLPdyn":
        label_set = sorted(set(labels.values()))
        label_map = {label: idx for idx, label in enumerate(label_set)}
    else:
        label_map = {str(i): i - 1 for i in range(1, label_count + 1)}

    snapshots: List[SnapshotData] = []
    for snapshot_index, graph in enumerate(graphs):
        adj_sp = sp.coo_matrix(nx.adjacency_matrix(graph))
        adj_dense = torch.tensor(adj_sp.toarray(), dtype=torch.float32, device=device)
        adj_norm_sp = graph_normalization(adj_sp)
        adj_norm_dense = torch.tensor(adj_norm_sp.toarray(), dtype=torch.float32, device=device)

        feature_matrix = compute_node2vec_features(graph, snapshot_index, config)
        features = torch.tensor(feature_matrix, dtype=torch.float32, device=device)

        if config.dataset_name == "DBLPdyn":
            label_list = [label_map[labels[snapshot_index][node]] for node in graph.nodes()]
        else:
            label_list = [label_map[labels[node]] for node in graph.nodes()]

        snapshots.append(
            SnapshotData(
                adj_sp=adj_sp,
                adj_dense=adj_dense,
                adj_norm_dense=adj_norm_dense,
                features=features,
                labels=torch.tensor(label_list, dtype=torch.long, device=device),
            )
        )

    return snapshots, label_count


def build_soft_community_graph(q: torch.Tensor, adj_dense: torch.Tensor) -> torch.Tensor:
    indicator = torch.argmax(q, dim=1)
    indicator_hat = F.one_hot(indicator, num_classes=q.size(1)).float()
    q_hat = indicator_hat * q
    comm_graph = q_hat.t() @ adj_dense @ q_hat
    comm_graph = comm_graph * (1 - torch.eye(comm_graph.size(0), device=comm_graph.device))
    denom = adj_dense.sum().clamp_min(1.0)
    return comm_graph / denom


def wrcf(graph: np.ndarray) -> gd.SimplexTree:
    require_gudhi()
    nx_graph = nx.from_numpy_array(graph)
    st = gd.SimplexTree()
    for node in nx_graph.nodes():
        st.insert([node], filtration=0.0)

    edge_weights = [weight for _, _, weight in nx_graph.edges.data("weight")]
    distinct_weights = np.unique(edge_weights)[::-1] if edge_weights else []
    for weight in distinct_weights:
        if weight <= 0:
            continue
        subgraph = nx_graph.edge_subgraph([(u, v) for u, v, w in nx_graph.edges.data("weight") if w >= weight])
        for clique in nx.find_cliques(subgraph):
            st.insert(clique, filtration=float(1.0 / max(weight, 1e-12)))
    return st


def wrcf_index(graph: np.ndarray, dim: int, card: int) -> np.ndarray:
    st = wrcf(graph)
    st.persistence()
    pairs = st.persistence_pairs()

    indices: List[int] = []
    persistence_scores: List[float] = []
    for simplex_birth, simplex_death in pairs:
        if len(simplex_birth) != dim + 1 or len(simplex_death) == 0:
            continue
        birth_vertices = np.array(simplex_birth)
        death_vertices = np.array(simplex_death)
        birth_pair = [
            simplex_birth[v]
            for v in np.unravel_index(
                np.argmax(graph[birth_vertices, :][:, birth_vertices]),
                [len(simplex_birth), len(simplex_birth)],
            )
        ]
        death_pair = [
            simplex_death[v]
            for v in np.unravel_index(
                np.argmax(graph[death_vertices, :][:, death_vertices]),
                [len(simplex_death), len(simplex_death)],
            )
        ]
        indices.extend(birth_pair)
        indices.extend(death_pair)
        persistence_scores.append(st.filtration(simplex_death) - st.filtration(simplex_birth))

    if persistence_scores:
        order = np.argsort(persistence_scores)
        indices = list(np.reshape(indices, [-1, 4])[order][::-1, :].flatten())
    indices = indices[: 4 * card] + [0 for _ in range(max(0, 4 * card - len(indices)))]
    return np.array(indices, dtype=np.int32)


class DeviceAwareWrcfLayer(nn.Module):
    def __init__(self, dim: int, card: int, device: torch.device) -> None:
        super().__init__()
        self.dim = dim
        self.card = card
        self.device = device

    def forward(self, graph: torch.Tensor) -> torch.Tensor:
        ids_np = wrcf_index(graph.detach().cpu().numpy(), self.dim, self.card)
        ids = torch.from_numpy(ids_np).long().to(graph.device)

        if self.dim > 0:
            indices = ids.view([2 * self.card, 2]).long()
            dgm = graph[indices[:, 0], indices[:, 1]].view(self.card, 2)
        else:
            indices = ids.view([2 * self.card, 2])[1::2, :].long()
            dgm = torch.cat(
                [
                    torch.zeros(self.card, 1, device=graph.device),
                    graph[indices[:, 0], indices[:, 1]].view(self.card, 1).float(),
                ],
                dim=1,
            )
        return dgm.to(self.device)


def safe_wasserstein(current: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    require_gudhi()
    return wasserstein_distance(
        current.double(),
        target.double(),
        order=1,
        enable_autodiff=True,
        keep_essential_parts=False,
    )


def diagram_pair_loss(
    current: Tuple[torch.Tensor, torch.Tensor],
    target_pair: Tuple[Optional[Tuple[torch.Tensor, torch.Tensor]], Optional[Tuple[torch.Tensor, torch.Tensor]]],
) -> torch.Tensor:
    losses: List[torch.Tensor] = []
    for target in target_pair:
        if target is None:
            continue
        losses.append(safe_wasserstein(current[0], target[0]))
        losses.append(safe_wasserstein(current[1], target[1]))
    if not losses:
        return torch.tensor(0.0, device=current[0].device)
    return torch.stack(losses).sum()


class CommunityTopoLoss(nn.Module):
    def __init__(
        self,
        card: int,
        targets: Tuple[Optional[Tuple[torch.Tensor, torch.Tensor]], Optional[Tuple[torch.Tensor, torch.Tensor]]],
        device: torch.device,
    ) -> None:
        super().__init__()
        self.targets = targets
        self.wrcf_dim0 = DeviceAwareWrcfLayer(dim=0, card=card, device=device)
        self.wrcf_dim1 = DeviceAwareWrcfLayer(dim=1, card=card, device=device)

    def forward(self, adj_dense: torch.Tensor, q: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, float]]:
        comm_graph = build_soft_community_graph(q, adj_dense)
        comm_dgm = (self.wrcf_dim0(comm_graph), self.wrcf_dim1(comm_graph))
        loss = diagram_pair_loss(comm_dgm, self.targets)
        stats = {"community_topo": float(loss.detach().cpu())}
        return loss, stats


def compute_reconstruction_loss(adj_dense: torch.Tensor, adj_pred: torch.Tensor) -> torch.Tensor:
    positive = adj_dense.sum().clamp_min(1.0)
    negative = torch.tensor(float(adj_dense.numel()), device=adj_dense.device) - positive
    pos_weight = negative / positive
    norm = torch.tensor(float(adj_dense.numel()), device=adj_dense.device) / ((negative * 2.0).clamp_min(1.0))
    weights = torch.ones_like(adj_dense)
    weights = torch.where(adj_dense > 0, pos_weight, weights)
    return norm * F.binary_cross_entropy(adj_pred, adj_dense, weight=weights)


def pretrain_snapshot(model: GAEMF, snapshot: SnapshotData, config: ExperimentConfig) -> List[Dict[str, float]]:
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    history: List[Dict[str, float]] = []

    model.train()
    for epoch in range(config.num_pretrain_epoch):
        use_mf = epoch >= config.start_mf
        adj_pred, z, q = model(snapshot.features, use_mf)
        recon = compute_reconstruction_loss(snapshot.adj_dense, adj_pred)
        cluster = torch.tensor(0.0, device=snapshot.adj_dense.device)
        if use_mf and q is not None:
            cluster = F.mse_loss(z, q @ model.cluster_centroid)
        loss = recon + config.alpha_pretrain * cluster

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(
            {
                "epoch": epoch,
                "loss": float(loss.detach().cpu()),
                "recon": float(recon.detach().cpu()),
                "cluster": float(cluster.detach().cpu()),
            }
        )
    return history


@torch.no_grad()
def extract_snapshot_state(
    model: GAEMF,
    snapshot: SnapshotData,
    wrcf_dim0: DeviceAwareWrcfLayer,
    wrcf_dim1: DeviceAwareWrcfLayer,
) -> Dict[str, torch.Tensor]:
    model.eval()
    _, _, q = model(snapshot.features, True)
    comm_graph = build_soft_community_graph(q, snapshot.adj_dense)
    return {
        "comm_dim0": wrcf_dim0(comm_graph).detach(),
        "comm_dim1": wrcf_dim1(comm_graph).detach(),
    }


def build_community_targets(
    states: Sequence[Dict[str, torch.Tensor]],
    index: int,
) -> Tuple[Optional[Tuple[torch.Tensor, torch.Tensor]], Optional[Tuple[torch.Tensor, torch.Tensor]]]:
    prev_state = states[index - 1] if index > 0 else None
    next_state = states[index + 1] if index < len(states) - 1 else None
    return (
        (prev_state["comm_dim0"], prev_state["comm_dim1"]) if prev_state is not None else None,
        (next_state["comm_dim0"], next_state["comm_dim1"]) if next_state is not None else None,
    )


def finetune_snapshot(
    model: GAEMF,
    snapshot: SnapshotData,
    topo_loss: CommunityTopoLoss,
    config: ExperimentConfig,
) -> List[Dict[str, float]]:
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    history: List[Dict[str, float]] = []

    model.train()
    for epoch in range(config.num_topo_epoch):
        adj_pred, z, q = model(snapshot.features, True)
        recon = compute_reconstruction_loss(snapshot.adj_dense, adj_pred)
        cluster = F.mse_loss(z, q @ model.cluster_centroid)
        topo, topo_stats = topo_loss(snapshot.adj_dense, q)
        loss = recon + config.alpha_topo * cluster + config.beta_topo * topo

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(
            {
                "epoch": epoch,
                "loss": float(loss.detach().cpu()),
                "recon": float(recon.detach().cpu()),
                "cluster": float(cluster.detach().cpu()),
                "community_topo": topo_stats["community_topo"],
            }
        )
    return history


def clustering_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.array(y_true, dtype=np.int64).copy()
    y_pred = np.array(y_pred, dtype=np.int64)
    voted = np.zeros_like(y_true)
    labels = np.unique(y_true)
    ordered = np.arange(labels.shape[0])
    for idx, label in enumerate(labels):
        y_true[y_true == label] = ordered[idx]
    bins = np.concatenate((np.unique(y_true), [np.max(y_true) + 1]), axis=0)
    for cluster in np.unique(y_pred):
        hist, _ = np.histogram(y_true[y_pred == cluster], bins=bins)
        winner = np.argmax(hist) if hist.size else 0
        voted[y_pred == cluster] = winner
    return float(metrics.accuracy_score(y_true, voted))


def modularity(adjacency_matrix: sp.coo_matrix, label_list: np.ndarray) -> float:
    labels = np.asarray(label_list, dtype=np.int64)
    num_classes = int(labels.max()) + 1 if labels.size else 0
    indicator = np.eye(num_classes)[labels] if num_classes else np.zeros((0, 0))
    adjacency = adjacency_matrix.toarray()
    m = np.sum(adjacency) / 2
    if m == 0:
        return 0.0
    degree = np.sum(adjacency, axis=1)
    modularity_matrix = adjacency - np.outer(degree, degree) / (2 * m)
    return float((1 / (2 * m)) * np.trace(indicator.T @ modularity_matrix @ indicator))


@torch.no_grad()
def evaluate_snapshot(model: GAEMF, snapshot: SnapshotData, config: ExperimentConfig) -> Dict[str, float]:
    model.eval()
    adj_pred, z, q = model(snapshot.features, True)
    if should_use_argmax_q(config):
        predicted = torch.argmax(q, dim=1).cpu().numpy()
    else:
        predicted = KMeans(n_clusters=config.n_cluster, n_init=20).fit_predict(z.cpu().numpy())
    labels = snapshot.labels.cpu().numpy()

    acc = clustering_accuracy(labels, predicted)
    nmi = metrics.normalized_mutual_info_score(labels, predicted)
    ari = metrics.adjusted_rand_score(labels, predicted)
    mod = modularity(snapshot.adj_sp, predicted)
    recon_acc = float((adj_pred > 0.5).eq(snapshot.adj_dense > 0).float().mean().cpu())
    return {"acc": acc, "nmi": float(nmi), "ari": float(ari), "modularity": float(mod), "recon_acc": recon_acc}


def train_model(config: ExperimentConfig) -> Dict[str, object]:
    require_gudhi()
    require_gensim()
    set_seed(config.seed)
    device = resolve_device(config.device)

    snapshots, label_count = load_dynamic_dataset(config, device)
    config = replace(config, n_cluster=resolve_n_cluster(config, label_count))

    save_dir = Path(config.save_dir) / config.dataset_name
    save_dir.mkdir(parents=True, exist_ok=True)

    models: List[GAEMF] = []
    pretrain_history: List[List[Dict[str, float]]] = []
    for snapshot in snapshots:
        model = GAEMF(snapshot.adj_norm_dense, snapshot.features.size(1), config).to(device)
        models.append(model)
        pretrain_history.append(pretrain_snapshot(model, snapshot, config))

    wrcf_dim0 = DeviceAwareWrcfLayer(dim=0, card=config.topo_cardinality, device=device)
    wrcf_dim1 = DeviceAwareWrcfLayer(dim=1, card=config.topo_cardinality, device=device)
    states = [extract_snapshot_state(model, snapshot, wrcf_dim0, wrcf_dim1) for model, snapshot in zip(models, snapshots)]

    finetune_history: List[List[Dict[str, float]]] = []
    metrics_by_snapshot: List[Dict[str, float]] = []
    for index, (model, snapshot) in enumerate(zip(models, snapshots)):
        topo_loss = CommunityTopoLoss(
            card=config.topo_cardinality,
            targets=build_community_targets(states, index),
            device=device,
        ).to(device)
        finetune_history.append(finetune_snapshot(model, snapshot, topo_loss, config))
        metrics_by_snapshot.append(evaluate_snapshot(model, snapshot, config))

    summary = {
        "config": asdict(config),
        "device": str(device),
        "metrics_by_snapshot": metrics_by_snapshot,
        "mean_metrics": {
            key: float(np.mean([entry[key] for entry in metrics_by_snapshot]))
            for key in metrics_by_snapshot[0].keys()
        },
    }

    with (save_dir / "summary.json").open("w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2)
    with (save_dir / "history.pkl").open("wb") as handle:
        pickle.dump({"pretrain": pretrain_history, "finetune": finetune_history}, handle)
    return summary


def print_summary(summary: Dict[str, object]) -> None:
    mean_metrics = summary["mean_metrics"]
    print(f"Device: {summary['device']}")
    for key in ["acc", "nmi", "ari", "modularity", "recon_acc"]:
        print(f"{key}: {mean_metrics[key]:.4f}")


In [ ]:
config = ExperimentConfig(
    dataset_name="highschool",
    max_snapshots=10,
    encoded_space_dim=30,
    structural_feature_dim=128,
    node2vec_walk_length=40,
    node2vec_num_walks=10,
    node2vec_window_size=10,
    node2vec_epochs=20,
    node2vec_p=1.0,
    node2vec_q=2.0,
    seed=7,
)

summary = train_model(config)
print_summary(summary)
summary
